# Treino interativo -- `MixrFlightEnv` + PPO (Stable-Baselines3)

Equivalente celula-a-celula de [`train.py`](../train.py) -- mesmo ambiente (`mixr_gym.MixrFlightEnv`), mesmo algoritmo (PPO), mesmo wrapper de achatamento (`flatten_obs.FlattenedObservation`). Ver [`README.md`](../README.md), secao "Notebook interativo", para como selecionar o kernel (`.venv` desta pasta) no VS Code.

**ORDEM IMPORTA -- leia antes de rodar:**

1. Rode as celulas **em ordem**, de cima pra baixo, na primeira vez.
2. **Nao abra o painel Variables/Data Viewer do VS Code** antes da celula de `import mixr_gym` (secao 2) -- ele injeta import de numpy/pandas no kernel por conta propria, e pode disparar a mesma armadilha da celula 1 (numpy antes de `mixr_gym`) por um caminho invisivel no notebook.
3. Se a asserção da celula 1 disparar (ou qualquer coisa parecer inconsistente porque uma celula rodou fora de ordem): **reinicie o kernel** (Restart) antes de tentar de novo -- so re-executar a celula NAO basta, o processo do kernel ja esta contaminado.

## 1. Bootstrap: localizar `mixr_gym` e a raiz do repositorio

In [ ]:
import os
import sys
import pathlib

# GUARDA (nao pular): o kernel deste notebook e um processo Python comum --
# se numpy/gymnasium ja tiverem sido importados por QUALQUER outro caminho
# antes desta celula (um arquivo de startup do IPython em
# ~/.ipython/profile_default/startup/, ou o painel Variables/Data Viewer do
# VS Code injetando codigo de introspeccao), a armadilha documentada em
# mixr_gym/__init__.py dispara: a PRIMEIRA chamada a reset() segfauta dentro
# de libstdc++, sem erro Python nenhum pra depurar. Esta linha transforma
# essa contaminacao silenciosa num erro CLARO, agora, em vez de um crash
# ilegivel depois.
_poluido = {"numpy", "gymnasium"} & set(sys.modules)
assert not _poluido, (
    f"numpy/gymnasium ja carregados no kernel ({sorted(_poluido)}) antes de "
    "'mixr_gym'. Nao basta re-rodar esta celula -- REINICIE O KERNEL "
    "(Restart) e rode as celulas em ordem, sem abrir o painel "
    "Variables/Data Viewer antes da proxima celula."
)


def _find_repo_root(marker=("dist", "python", "mixr_gym")) -> pathlib.Path:
    """Busca ascendente pela raiz do repositorio, a partir do cwd do kernel.

    Nao da pra confiar em PYTHONPATH de ambiente aqui -- o VS Code lanca o
    kernel Jupyter direto (`<venv>/bin/python -m ipykernel_launcher`), sem
    passar pelo Makefile que normalmente seta essa variavel. O marcador
    'dist/python/mixr_gym' e o MESMO que o alvo 'check-root' do Makefile
    desta pasta confere.
    """
    candidatos = [pathlib.Path.cwd(), *pathlib.Path.cwd().resolve().parents]
    for candidato in candidatos:
        if candidato.joinpath(*marker).exists():
            return candidato
    raise RuntimeError(
        "nao achei 'dist/python/mixr_gym' subindo a partir de "
        f"{pathlib.Path.cwd()}. Rode 'make configure && make sdk && make "
        "build && make install' na raiz do repositorio primeiro, ou confira "
        "a configuracao 'jupyter.notebookFileRoot' do VS Code."
    )


REPO_ROOT = _find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "dist" / "python"))
sys.path.insert(0, str(REPO_ROOT / "src" / "poc" / "rl-training"))

# MixrFlightEnv resolve 'scenario_path' por caminho relativo -- mesma
# convencao de todo binario deste projeto (ver CLAUDE.md/src/rl/README.md).
# O cwd do kernel Jupyter depende de 'jupyter.notebookFileRoot' do VS Code,
# entao fixamos explicitamente na raiz do repo em vez de confiar no default.
os.chdir(REPO_ROOT)
print(f"raiz do repositorio: {REPO_ROOT}")
print(f"cwd:                 {pathlib.Path.cwd()}")

## 2. Importar `mixr_gym` (PRIMEIRO import de terceiros)

In [ ]:
# ARMADILHA CONFIRMADA (nao redescobrir -- ver mixr_gym/__init__.py e
# src/rl/README.md, secao "Limites conhecidos"): 'mixr_gym' TEM DE ser o
# PRIMEIRO import deste processo que toca numpy/gymnasium/stable_baselines3
# -- e por isso que esta celula vem logo apos o bootstrap, antes de
# QUALQUER outro import de terceiros (inclusive na celula seguinte).
from mixr_gym import MixrFlightEnv
from mixr_gym.env import DEFAULT_PLAYER, DEFAULT_SCENARIO

print("mixr_gym importado com seguranca.")

## 3. Demais imports

In [ ]:
from flatten_obs import FlattenedObservation

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback

## 4. Parametros do treino

Equivalente interativo dos argumentos de `train.py --help` -- edite e re-execute esta celula (e as seguintes) para experimentar.

In [ ]:
DEFAULT_SCENARIO

In [ ]:
SCENARIO = "src/poc/rl-training/configs/scenario_rl.edl" # DEFAULT_SCENARIO
PLAYER = 'falcon1' # DEFAULT_PLAYER
MAX_EPISODE_STEPS = 2000
SEED = 0
# Ancorado em REPO_ROOT, NUNCA relativo a cwd 'solto' -- o mesmo motivo
# documentado em train.py (DEFAULT_OUT_DIR): uma string tipo
# pathlib.Path("./runs") escreveria em <raiz-do-repo>/runs/, fora do
# .gitignore desta pasta, porque a celula de bootstrap ja fez os.chdir
# para a raiz do repositorio (necessario para MixrFlightEnv resolver
# 'scenario_path'). MEDIDO acontecendo antes deste fix.
OUT_DIR = REPO_ROOT / "src" / "poc" / "rl-training" / "runs"

print(f"cenario:  {SCENARIO}")
print(f"player:   {PLAYER}")

## 5. Construir o ambiente

In [ ]:
env = MixrFlightEnv(
    scenario_path=SCENARIO,
    player_name=PLAYER,
    max_episode_steps=MAX_EPISODE_STEPS,
)
# 'MlpPolicy' (secao 7) precisa de observacao plana -- ver o docstring de
# FlattenedObservation em flatten_obs.py para o "porque" (o contrato .onnx
# de producao e float32[1,28], nao o Dict/Discrete nativo de MixrFlightEnv).
env = FlattenedObservation(env)
print("MixrFlightEnv + FlattenedObservation prontos.")

## 6. Inspecionar observacao/acao

`train.py` (CLI) nao expõe isto -- e o principal ganho de rodar como notebook: ver a forma dos espacos e alguns passos manuais antes de comprometer um treino longo.

In [ ]:
print(f"observation_space: {env.observation_space}")
print(f"action_space:      {env.action_space}")

obs, info = env.reset(seed=SEED)
print(f"\nobservacao inicial (achatada, shape={obs.shape}):\n{obs}")

for _ in range(3):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"acao={action}  reward={reward:.3f}  terminated={terminated}")

## 7. Criar o modelo PPO

In [ ]:
model = PPO(
    "MlpPolicy",
    env,
    seed=SEED,
    verbose=1,
)

## 8. Checkpoints periodicos

O episodio inteiro depende do JSBSim integrando em tempo real de CPU -- perder progresso por um kernel travado/reiniciado custa caro. Ver `train.py` para o equivalente em script (que tambem salva no `Ctrl+C`, algo que nao se aplica a um notebook).

In [ ]:
checkpoints_dir = OUT_DIR / "checkpoints"
checkpoints_dir.mkdir(parents=True, exist_ok=True)

checkpoint_callback = CheckpointCallback(
    save_freq=20_000,
    save_path=str(checkpoints_dir),
    name_prefix=f"ppo_{PLAYER}",
)

## 9. Treinar

**Re-execute esta celula quantas vezes quiser** -- `reset_num_timesteps=False` faz o contador de passos continuar de onde parou (o mesmo `model` acumula treino entre execucoes), em vez de reiniciar do zero a cada rodada.

In [ ]:
TIMESTEPS_PER_RUN = 20_000

model.learn(
    total_timesteps=TIMESTEPS_PER_RUN,
    reset_num_timesteps=False,
    callback=checkpoint_callback,
)

## 10. Salvar um checkpoint manual

In [ ]:
final_path = OUT_DIR / f"ppo_{PLAYER}"
model.save(str(final_path))
print(f"checkpoint salvo em {final_path}.zip")

## 11. Exportar para `.onnx` (opcional)

Mesmo contrato de `make export` (`tools/export_onnx.py --sb3 ...`) -- ver [`README.md`](../README.md) e `models/flight/docs/POLITICAS.md` para como apontar o `.onnx` resultante para `src/poc/onnx-policy` ou para qualquer `treeFile:` com um no `( OnnxPolicy )`.

In [ ]:
onnx_out = OUT_DIR / "politica.onnx"
export_script = (
    REPO_ROOT / "src" / "poc" / "rl-training" / "tools" / "export_onnx.py"
)
cmd = (
    f'PYTHONPATH={REPO_ROOT / "dist" / "python"} {sys.executable} '
    f'{export_script} --sb3 {final_path}.zip -o {onnx_out}'
)
print(cmd)
!{cmd}

## 12. Encerrar

In [ ]:
env.close()